# 2D Slot Attention Evaluation

Evaluates a trained `SlotClassifier2D` on two tasks:

1. **Volume Segmentation** — Adjusted Rand Index (ARI) and slot mask visualisations (BraTS)
2. **Object Classification** — per-target accuracy / precision / recall / F1:
   - *BraTS*: tumour type, volume bin, centroid-Y, centroid-X
   - *BrainWear*: EORTC PRO outcome prediction from slot features

**Set `CHECKPOINT_PATH`** in the Configuration cell before running.

In [ ]:
%matplotlib inline
import sys, os

FYP_ROOT = "/path/to/BrainWear_Kareem/FYP"
if FYP_ROOT not in sys.path:
    sys.path.insert(0, FYP_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from torch.utils.data import DataLoader
from sklearn.metrics import (
    adjusted_rand_score, accuracy_score,
    precision_score, recall_score, f1_score,
)
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D
from datasets.brats2020_png import BraTS2020PNGDataset, batch_seg_to_slot_targets_2d
from datasets.brainwear_png import BrainWearPNGDataset
from utils.utils import compute_cost_matrix_2d, hungarian_algorithm
from utils.characterisations import TumourCharacterisationSmall2D
from utils.constants import (
    VOLUME_2D_SMALL_THRESHOLD,
    VOLUME_2D_MEDIUM_THRESHOLD,
    SPATIAL_MIDPOINT,
)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
CHECKPOINTS = [
    {"path": os.path.join(FYP_ROOT, "slot_attention/training_2d/models/checkpoints/brats_png_v14a_0.15_test/ckpt.pt"),
     "label": "Fully supervised"},
    {"path": os.path.join(FYP_ROOT, "slot_attention/training_2d/models/checkpoints/brats_png_v17_weak_0.15_new_norm/ckpt.pt"),
     "label": "Weakly supervised"},
    {"path": os.path.join(FYP_ROOT, "slot_attention/training_2d/models/checkpoints/brats_png_v19a_entropy/ckpt.pt"),
     "label": "Weakly supervised + spatial dice matching"},
    # add more checkpoints here as needed
]

# First checkpoint drives all non-IoU/Dice sections (ARI, classification, BrainWear)
CHECKPOINT_PATH = CHECKPOINTS[0]["path"]
EXPERIMENT_NAME = os.path.basename(os.path.dirname(CHECKPOINT_PATH))

BASE_DIR             = "/path/to/BrainWear_Kareem"
BRATS_DATA_DIR       = os.path.join(BASE_DIR, "Processed_BraTS2020_TrainingData_PNG")
BRAINWEAR_DATA_DIR   = os.path.join(BASE_DIR, "Processed_Brainwear_PNG_fixed_norm")
BRAINWEAR_SCORES_CSV = os.path.join(BASE_DIR, "eortc_scores.csv")
SEG_PLOT_DIR         = os.path.join(FYP_ROOT, "eval", "segmentation_plots")
os.makedirs(SEG_PLOT_DIR, exist_ok=True)

def _plot_path(name: str) -> str:
    prefix = f"{EXPERIMENT_NAME}_" if EXPERIMENT_NAME else ""
    return os.path.join(SEG_PLOT_DIR, f"{prefix}{name}")

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
N_VIS      = 5
N_BINS     = 3
SEED       = 42

EVAL_PATIENT_FRACTION = 0.15

EORTC_SCORES = ["QL2", "PF2", "EF", "CF"]

print(f"Device: {DEVICE}")
print(f"Checkpoints: {len(CHECKPOINTS)}")
for c in CHECKPOINTS:
    exists = os.path.exists(c["path"])
    print(f"  {'[OK]' if exists else '[MISSING]'} {c['label']}: {os.path.basename(os.path.dirname(c['path']))}")

In [ ]:
# ── Model loading ────────────────────────────────────────────────────────────
def _hp(hp, key, default):
    return hp.get(key, default) if isinstance(hp, dict) else getattr(hp, key, default)


def load_slot_model_2d(path: str, device: torch.device) -> SlotClassifier2D:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    hp   = ckpt.get("hyperparameters", {})
    m = SlotClassifier2D(
        in_shape=(_hp(hp, "in_channels", 1), _hp(hp, "input_h", 240), _hp(hp, "input_w", 240)),
        width=_hp(hp, "width", 64),
        num_slots=_hp(hp, "num_slots", 5),
        slot_dim=_hp(hp, "slot_dim", 64),
        routing_iters=_hp(hp, "routing_iters", 7),
        temperature=_hp(hp, "temp", 0.5),
        encoder_depth=_hp(hp, "encoder_depth", 4),
        enc3_init_skip=_hp(hp, "enc3_init_skip", False),
        use_mask_pool_classifier=_hp(hp, "use_mask_pool_classifier", False),
    )
    m.load_state_dict(ckpt["model_state_dict"])
    m.to(device).eval()
    print(f"  Loaded '{os.path.basename(os.path.dirname(path))}' (epoch {ckpt.get('epoch', '?')})")
    return m


print("Loading models...")
models = {c["label"]: load_slot_model_2d(c["path"], DEVICE) for c in CHECKPOINTS}

# First model drives the rest of the notebook (ARI, classification, BrainWear)
model     = list(models.values())[0]
NUM_SLOTS = model._num_slots
print(f"Primary model: '{CHECKPOINTS[0]['label']}'  num_slots={NUM_SLOTS}")

In [ ]:
# ── BraTS dataset ────────────────────────────────────────────────────────────
# Restrict to the first EVAL_PATIENT_FRACTION of patients (alphabetical order)
# — these were held out from training.
_all_brats_patients = sorted([
    p for p in os.listdir(BRATS_DATA_DIR)
    if os.path.isdir(os.path.join(BRATS_DATA_DIR, p))
])
_n_brats_total = len(_all_brats_patients)
_brats_eval_patients = set(
    _all_brats_patients[:max(1, int(_n_brats_total * EVAL_PATIENT_FRACTION))]
)

brats_dataset = BraTS2020PNGDataset(BRATS_DATA_DIR, is_train=False)
brats_dataset.samples = [
    (t2, seg) for t2, seg in brats_dataset.samples
    if os.path.basename(os.path.dirname(t2)) in _brats_eval_patients
]
brats_loader  = DataLoader(brats_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
print(f"BraTS 2D: {len(brats_dataset)} samples "
      f"(first {len(_brats_eval_patients)}/{_n_brats_total} patients, {EVAL_PATIENT_FRACTION:.0%})")

---
## Section 1 — Volume Segmentation

In [ ]:
# ── IoU and Dice computation for all checkpoints ─────────────────────────────
CLASS_NAMES = {0: "Background", 1: "NCR/NET", 2: "ED", 3: "ET"}


def _compute_iou_dice(m, loader, device):
    iou  = {c: [] for c in range(4)}
    dice = {c: [] for c in range(4)}
    num_slots = m._num_slots
    m.eval()
    with torch.no_grad():
        for t2_batch, seg_batch in tqdm(loader, desc="IoU/Dice", leave=False):
            t2_batch = t2_batch.to(device)
            _, _, masks, _, _ = m(t2_batch)
            slot_assign = masks.squeeze(2).cpu().argmax(dim=1)  # (B, H, W)
            for i in range(t2_batch.shape[0]):
                seg_np = seg_batch[i].numpy()
                for cls in range(4):
                    gt_mask = seg_np == cls
                    if not gt_mask.any():
                        continue
                    best_iou = best_dice = 0.0
                    for k in range(num_slots):
                        pred_mask = slot_assign[i].numpy() == k
                        inter = (gt_mask & pred_mask).sum()
                        union = (gt_mask | pred_mask).sum()
                        denom = gt_mask.sum() + pred_mask.sum()
                        iou_k  = inter / union  if union  > 0 else 0.0
                        dice_k = 2 * inter / denom if denom > 0 else 0.0
                        if iou_k > best_iou:
                            best_iou, best_dice = iou_k, dice_k
                    iou[cls].append(best_iou)
                    dice[cls].append(best_dice)
    return iou, dice


ckpt_iou_dice = {}
for label, m in models.items():
    print(f"Computing IoU/Dice: {label}")
    iou, dice = _compute_iou_dice(m, brats_loader, DEVICE)
    ckpt_iou_dice[label] = {
        "iou":  iou,
        "dice": dice,
        "mean_iou_bg":    float(np.mean([np.mean(iou[c])  for c in range(4)])),
        "mean_iou_nobg":  float(np.mean([np.mean(iou[c])  for c in range(1, 4)])),
        "mean_dice_bg":   float(np.mean([np.mean(dice[c]) for c in range(4)])),
        "mean_dice_nobg": float(np.mean([np.mean(dice[c]) for c in range(1, 4)])),
    }

# Print summary table
print(f"\n{'Checkpoint':<42} {'mIoU+BG':>8} {'mIoU':>8} {'mDice+BG':>10} {'mDice':>8}")
print("-" * 80)
for label, r in ckpt_iou_dice.items():
    print(f"{label:<42} {r['mean_iou_bg']:>8.3f} {r['mean_iou_nobg']:>8.3f} "
          f"{r['mean_dice_bg']:>10.3f} {r['mean_dice_nobg']:>8.3f}")

# Preserve single-checkpoint variables for the rest of the notebook
first = list(ckpt_iou_dice.values())[0]
per_class_iou  = first["iou"]
per_class_dice = first["dice"]
mean_iou_bg    = first["mean_iou_bg"]
mean_iou_nobg  = first["mean_iou_nobg"]
mean_dice_bg   = first["mean_dice_bg"]
mean_dice_nobg = first["mean_dice_nobg"]

In [ ]:
# ── IoU / Dice grouped bar chart: all checkpoints on the same graph ───────────
ckpt_labels = list(ckpt_iou_dice.keys())
n_ckpts     = len(ckpt_labels)
n_classes   = 4
class_labels = [CLASS_NAMES[c] for c in range(n_classes)]
colours      = plt.cm.tab10(np.linspace(0, 0.9, n_ckpts))

x          = np.arange(n_classes)
bar_w      = 0.7 / n_ckpts
offsets    = np.linspace(-(0.7 - bar_w) / 2, (0.7 - bar_w) / 2, n_ckpts)

fig, axes = plt.subplots(1, 2, figsize=(max(10, 3 * n_ckpts + 3), 4.5))

for ax, metric, title, ylabel in [
    (axes[0], "iou",  "IoU per class",  "IoU"),
    (axes[1], "dice", "Dice per class", "Dice"),
]:
    for ci, (label, col, off) in enumerate(zip(ckpt_labels, colours, offsets)):
        vals = [np.mean(ckpt_iou_dice[label][metric][c]) for c in range(n_classes)]
        stds = [np.std( ckpt_iou_dice[label][metric][c]) for c in range(n_classes)]
        bars = ax.bar(x + off, vals, bar_w, label=label, color=col,
                      yerr=stds, capsize=3, zorder=3)
        for bar, v, s in zip(bars, vals, stds):
            ax.text(bar.get_x() + bar.get_width() / 2, v + s + 0.02,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=9, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, fontsize=9)
    ax.set_ylim(0, 1.55)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.yaxis.grid(True, linestyle=":", alpha=0.4, zorder=0)
    ax.legend(fontsize=7)

plt.suptitle("Volume Segmentation: IoU & Dice by Checkpoint (BraTS)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(SEG_PLOT_DIR, "iou_dice_comparison_2d.png"), bbox_inches="tight")
plt.show()

# ── Summary: mean IoU / Dice excl. background ────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(max(6, n_ckpts * 2 + 2), 4))
for ax2, key, ylabel in [
    (axes2[0], "mean_iou_nobg",  "Mean IoU (excl. background)"),
    (axes2[1], "mean_dice_nobg", "Mean Dice (excl. background)"),
]:
    vals = [ckpt_iou_dice[l][key] for l in ckpt_labels]
    bars = ax2.bar(range(n_ckpts), vals, color=colours, zorder=3)
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.001,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax2.set_xticks(range(n_ckpts))
    ax2.set_xticklabels(ckpt_labels, fontsize=8, rotation=15, ha="right")
    ax2.set_ylim(0, max(vals) * 1.35 + 0.005)
    ax2.set_ylabel(ylabel)
    ax2.yaxis.grid(True, linestyle=":", alpha=0.4, zorder=0)

plt.suptitle("Summary: Mean IoU & Dice excl. Background", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(SEG_PLOT_DIR, "iou_dice_summary_2d.png"), bbox_inches="tight")
plt.show()

---
## Section 2 — Object Classification: BraTS

For each BraTS sample the model's predicted slots are matched to ground-truth objects
via the Hungarian algorithm.  For each matched non-background tumour pair we evaluate:
- **Tumour type** (4-class: background / NCR / ED / ET)
- **Volume** (3-class: small / medium / large, using dataset percentile thresholds)
- **Centroid-Y** and **Centroid-X** (binary left/right split at 0.5)

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────
def bin_volume_2d(v: float) -> int:
    """Small / medium / large using 2D dataset percentile thresholds."""
    if v < VOLUME_2D_SMALL_THRESHOLD:
        return 0
    if v < VOLUME_2D_MEDIUM_THRESHOLD:
        return 1
    return 2


def bin_spatial(c: float) -> int:
    """Binary spatial bin: 0 = low half, 1 = high half."""
    return 0 if c < SPATIAL_MIDPOINT else 1


def compute_metrics(y_true, y_pred, average="macro") -> dict:
    yt, yp = np.array(y_true), np.array(y_pred)
    return {
        "Accuracy":  float(accuracy_score(yt, yp)),
        "Precision": float(precision_score(yt, yp, average=average, zero_division=0)),
        "Recall":    float(recall_score(yt, yp, average=average, zero_division=0)),
        "F1":        float(f1_score(yt, yp, average=average, zero_division=0)),
    }

In [ ]:
# ── Hungarian matching + metric collection for all checkpoints ────────────────
def _compute_classification(m, loader, device, num_slots):
    y_true_type,  y_pred_type  = [], []
    y_true_vol,   y_pred_vol   = [], []
    y_true_centy, y_pred_centy = [], []
    y_true_centx, y_pred_centx = [], []
    gt_centy_raw, pred_centy_raw = [], []
    gt_centx_raw, pred_centx_raw = [], []

    H_img = W_img = None
    grid_y = grid_x = None

    m.eval()
    with torch.no_grad():
        for t2_batch, seg_batch in tqdm(loader, desc="BraTS classification", leave=False):
            B = t2_batch.shape[0]
            t2_batch = t2_batch.to(device)
            _, _, masks, _, output = m(t2_batch)
            masks_cpu = masks.squeeze(2).cpu()
            out_cpu   = output.cpu()

            if H_img is None:
                H_img, W_img = masks_cpu.shape[-2], masks_cpu.shape[-1]
                grid_y = torch.linspace(0, 1, H_img)
                grid_x = torch.linspace(0, 1, W_img)

            gt       = batch_seg_to_slot_targets_2d(seg_batch, num_slots)
            cost_mat = compute_cost_matrix_2d(out_cpu, gt, num_slots, coord_dim=2)
            _, indices = hungarian_algorithm(cost_mat)

            for b in range(B):
                for k in range(num_slots):
                    pi, gi  = indices[b, 0, k].item(), indices[b, 1, k].item()
                    gt_slot = gt[b, gi]
                    gt_cls  = gt_slot[:4]
                    if gt_cls.sum() == 0 or gt_cls.argmax().item() == 0:
                        continue

                    y_true_type.append(gt_cls.argmax().item())
                    y_pred_type.append(out_cpu[b, pi, :4].softmax(dim=0).argmax().item())

                    mask      = masks_cpu[b, pi]
                    pred_area = mask.sum().item() / (H_img * W_img)
                    y_true_vol.append(bin_volume_2d(gt_slot[6].item()))
                    y_pred_vol.append(bin_volume_2d(pred_area))

                    total_mass     = mask.sum().clamp(min=1e-6)
                    pred_centy_val = (mask * grid_y[:, None]).sum() / total_mass
                    pred_centx_val = (mask * grid_x[None, :]).sum() / total_mass
                    y_true_centy.append(bin_spatial(gt_slot[4].item()))
                    y_pred_centy.append(bin_spatial(pred_centy_val.item()))
                    y_true_centx.append(bin_spatial(gt_slot[5].item()))
                    y_pred_centx.append(bin_spatial(pred_centx_val.item()))
                    gt_centy_raw.append(gt_slot[4].item())
                    pred_centy_raw.append(pred_centy_val.item())
                    gt_centx_raw.append(gt_slot[5].item())
                    pred_centx_raw.append(pred_centx_val.item())

    return dict(
        y_true_type=y_true_type, y_pred_type=y_pred_type,
        y_true_vol=y_true_vol,   y_pred_vol=y_pred_vol,
        y_true_centy=y_true_centy, y_pred_centy=y_pred_centy,
        y_true_centx=y_true_centx, y_pred_centx=y_pred_centx,
        gt_centy_raw=gt_centy_raw, pred_centy_raw=pred_centy_raw,
        gt_centx_raw=gt_centx_raw, pred_centx_raw=pred_centx_raw,
    )


ckpt_classification = {}
for label, m in models.items():
    print(f"Computing classification: {label}")
    ckpt_classification[label] = _compute_classification(m, brats_loader, DEVICE, m._num_slots)
    print(f"  Matched tumour pairs: {len(ckpt_classification[label]['y_true_type'])}")

# Preserve first-checkpoint variables for backward compat
_first_cls = list(ckpt_classification.values())[0]
y_true_type,  y_pred_type  = _first_cls["y_true_type"],  _first_cls["y_pred_type"]
y_true_vol,   y_pred_vol   = _first_cls["y_true_vol"],   _first_cls["y_pred_vol"]
y_true_centy, y_pred_centy = _first_cls["y_true_centy"], _first_cls["y_pred_centy"]
y_true_centx, y_pred_centx = _first_cls["y_true_centx"], _first_cls["y_pred_centx"]
gt_centy_raw,  pred_centy_raw = _first_cls["gt_centy_raw"],  _first_cls["pred_centy_raw"]
gt_centx_raw,  pred_centx_raw = _first_cls["gt_centx_raw"],  _first_cls["pred_centx_raw"]

In [ ]:
# ── BraTS object classification: grouped bar charts per checkpoint ─────────────
_target_pairs = [
    ("Tumour Type", "y_true_type",  "y_pred_type"),
    ("Area",      "y_true_vol",   "y_pred_vol"),
    ("Centroid-Y",  "y_true_centy", "y_pred_centy"),
    ("Centroid-X",  "y_true_centx", "y_pred_centx"),
]
_target_names = [t[0] for t in _target_pairs]

ckpt_cls_results = {
    label: {name: compute_metrics(d[yt], d[yp]) for name, yt, yp in _target_pairs}
    for label, d in ckpt_classification.items()
}

_cls_labels = list(ckpt_classification.keys())
_n          = len(_cls_labels)
_colours    = plt.cm.tab10(np.linspace(0, 0.9, _n))
_bar_w      = 0.7 / _n
_x          = np.arange(len(_target_names))
_offsets    = np.linspace(-(_bar_w * (_n - 1)) / 2, (_bar_w * (_n - 1)) / 2, _n)

for metric in ["Accuracy", "Precision", "Recall", "F1"]:
    fig, ax = plt.subplots(figsize=(max(7, _n * 2 + 3), 4))
    for ci, (label, col, off) in enumerate(zip(_cls_labels, _colours, _offsets)):
        vals = [ckpt_cls_results[label][t][metric] for t in _target_names]
        bars = ax.bar(_x + off, vals, _bar_w, label=label, color=col, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=8, rotation=90)
    ax.set_xticks(_x)
    ax.set_xticklabels(_target_names, fontsize=9)
    ax.set_ylim(0, 1.45)
    ax.set_ylabel(metric)
    ax.set_title(f"Object Classification: {metric} (BraTS)")
    ax.yaxis.grid(True, linestyle=":", alpha=0.5, zorder=0)
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(SEG_PLOT_DIR, f"brats_2d_{metric.lower()}_comparison.png"), bbox_inches="tight")
    plt.show()

# Print summary table
for label, results in ckpt_cls_results.items():
    print(f"\n{label}")
    print(f"  {'Target':<15} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>6}")
    for t, m in results.items():
        print(f"  {t:<15} {m['Accuracy']:>9.3f} {m['Precision']:>10.3f} {m['Recall']:>8.3f} {m['F1']:>6.3f}")

# Backward compat
brats_results = ckpt_cls_results[_cls_labels[0]]

In [ ]:
# ── Centroid prediction scatter: all checkpoints ─────────────────────────────
_cls_labels = list(ckpt_classification.keys())
_n          = len(_cls_labels)
_colours    = plt.cm.tab10(np.linspace(0, 0.9, _n))

fig, axes = plt.subplots(2, _n, figsize=(_n * 3.5, 7), sharex=True, sharey=True,
                          squeeze=False)

for ci, (label, col) in enumerate(zip(_cls_labels, _colours)):
    d      = ckpt_classification[label]
    gt_y   = np.array(d["gt_centy_raw"]);   pred_y = np.array(d["pred_centy_raw"])
    gt_x   = np.array(d["gt_centx_raw"]);   pred_x = np.array(d["pred_centx_raw"])
    mae_y  = np.abs(pred_y - gt_y).mean()
    mae_x  = np.abs(pred_x - gt_x).mean()

    for row, (gt, pred, mae) in enumerate([(gt_y, pred_y, mae_y), (gt_x, pred_x, mae_x)]):
        ax = axes[row, ci]
        ax.scatter(gt, pred, s=6, alpha=0.3, color=col)
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_aspect("equal")
        ax.set_title(label if row == 0 else "", fontsize=8)
        ax.text(0.05, 0.92, f"MAE={mae:.3f}", transform=ax.transAxes, fontsize=8,
                va="top", color="black")
        ax.grid(True, linestyle=":", alpha=0.4)
        if ci == 0:
            ax.set_ylabel(["Pred centroid Y\n(anterior-posterior)",
                           "Pred centroid X\n(left-right)"][row], fontsize=8)
        if row == 1:
            ax.set_xlabel("GT centroid", fontsize=8)

plt.suptitle("Centroid Prediction: GT vs Predicted (BraTS)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(SEG_PLOT_DIR, "centroid_scatter_comparison_2d.png"), bbox_inches="tight")
plt.show()

# Print MAE / L2 table
print(f"\n{'Checkpoint':<42} {'MAE-Y':>8} {'MAE-X':>8} {'L2':>8}")
print("-" * 72)
for label, d in ckpt_classification.items():
    gy, py = np.array(d["gt_centy_raw"]), np.array(d["pred_centy_raw"])
    gx, px = np.array(d["gt_centx_raw"]), np.array(d["pred_centx_raw"])
    l2 = np.sqrt((py - gy) ** 2 + (px - gx) ** 2).mean()
    print(f"{label:<42} {np.abs(py-gy).mean():>8.4f} {np.abs(px-gx).mean():>8.4f} {l2:>8.4f}")

# Backward compat
gt_y   = np.array(_first_cls["gt_centy_raw"]);   pred_y = np.array(_first_cls["pred_centy_raw"])
gt_x   = np.array(_first_cls["gt_centx_raw"]);   pred_x = np.array(_first_cls["pred_centx_raw"])